# Step 2 — Discovery Pass (Google Books · HathiTrust · DPLA)

**Purpose:** For each person in the database, search by canonical name and name variants across three external APIs and log every hit to the `sources_discovered` table. This is a *search-and-log step only* — no full-text download happens here.

**Why three APIs?**

- **Google Books** — large index of book snippets; useful as *pointers* to volumes that may be available on Internet Archive in full. Snippets themselves are too short (1–2 sentences) to use as text sources directly.
- **HathiTrust** — strong 19th-century full-text coverage; catalog search surfaces volumes from research libraries. Full-text page-level search is not publicly API-accessible, so we use catalog-level volume discovery here.
- **DPLA** — Digital Public Library of America aggregates items from state historical societies, county libraries, and universities. Frequently surfaces local newspaper histories and biographical materials not indexed elsewhere.

After this step runs, use `step2_5_corpus_curation.ipynb` to review results and build the download manifest for Step 3a.

In [1]:
import sys, json, time, os
from pathlib import Path
import requests
from dotenv import load_dotenv
from tqdm import tqdm

sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, pending_persons, log_source, name_variants_for

load_dotenv('../.env')
conn = get_connection()

GOOGLE_BOOKS_KEY = os.getenv("GOOGLE_BOOKS_API_KEY", "")
DPLA_KEY = os.getenv("DPLA_API_KEY", "")

if not GOOGLE_BOOKS_KEY:
    print("WARNING: No GOOGLE_BOOKS_API_KEY in .env — Google Books step will be skipped")
if not DPLA_KEY:
    print("WARNING: No DPLA_API_KEY in .env — DPLA step will be skipped")

## 2a — Google Books API

In [2]:
def search_google_books(name, api_key, max_results=10):
    """
    Search Google Books for a person's name. Returns list of dicts with
    title, volumeId, snippet, infoLink.
    
    Note: Google Books API returns at most ~40 results per query. Snippets
    are short (usually 1-2 sentences). Use these as pointers to find volumes
    in Internet Archive, not as text sources.
    """
    if not api_key:
        return []
    
    # Wrap in quotes for exact phrase search
    query = f'"{name}"'
    
    try:
        resp = requests.get(
            "https://www.googleapis.com/books/v1/volumes",
            params={
                "q": query,
                "maxResults": max_results,
                "printType": "books",
                "key": api_key,
            },
            timeout=20
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"    Google Books error: {e}")
        return []
    
    results = []
    for item in data.get("items", []):
        info = item.get("volumeInfo", {})
        search_info = item.get("searchInfo", {})
        results.append({
            "item_id": item.get("id"),
            "title": info.get("title", ""),
            "authors": ", ".join(info.get("authors", [])),
            "published_date": info.get("publishedDate", ""),
            "snippet": search_info.get("textSnippet", ""),
            "info_link": info.get("infoLink", ""),
            "preview_link": info.get("previewLink", ""),
        })
    return results


def run_google_books_for_person(conn, research_id, canonical_name, variants, api_key):
    """Run Google Books search for all name variants. Returns count of sources logged."""
    all_names = [canonical_name] + variants
    count = 0
    for search_name in all_names[:3]:  # limit variants to avoid burning quota
        results = search_google_books(search_name, api_key)
        for r in results:
            sid = log_source(
                conn, research_id,
                source_type='google_books',
                title=r['title'],
                url=r['info_link'],
                item_id=r['item_id'],
                snippet=r['snippet'][:500] if r['snippet'] else None,
                relevance_note=f"authors: {r['authors']}; published: {r['published_date']}",
                discovery_step='2a_google_books',
            )
            if sid:
                count += 1
        time.sleep(0.5)
    return count

## 2b — HathiTrust Full-Text Search

In [3]:
def search_hathitrust_catalog(name, max_results=20):
    """
    Use HathiTrust's catalog search to find volumes mentioning a person.
    This uses the documented HathiTrust Bibliographic API.
    For full-text search across pages, use Internet Archive in Step 3a instead —
    HathiTrust's full-text search UI is not publicly API-accessible.
    
    Returns list of {htid, title, url, pub_date, description} dicts.
    """
    try:
        # HathiTrust doesn't have a public full-text search JSON API.
        # Use their catalog search to find relevant volumes by subject/title.
        # Then cross-reference these volumes on Internet Archive for actual text.
        resp = requests.get(
            "https://catalog.hathitrust.org/Search/Home",
            params={
                "lookfor": f'"{name}"',
                "type": "AllFields",
                "filter[]": "format:Book",
                "view": "list",
                "sort": "relevance",
                "limit": str(max_results),
            },
            headers={"User-Agent": "RailroadTiesPipeline/1.0"},
            timeout=30
        )
        resp.raise_for_status()
        
        import re
        text = resp.text
        results = []
        
        # Extract record IDs and titles from catalog HTML
        # HathiTrust catalog HTML contains links like /Record/001234567
        records = re.findall(r'/Record/(\d+)[^"]*"[^>]*>([^<]+)', text)
        seen = set()
        for record_id, title in records[:max_results]:
            if record_id in seen or not title.strip():
                continue
            seen.add(record_id)
            results.append({
                "htid": f"ht:{record_id}",
                "title": title.strip(),
                "url": f"https://catalog.hathitrust.org/Record/{record_id}",
                "item_id": record_id,
            })
        
        return results
    except Exception as e:
        print(f"    HathiTrust catalog error: {e}")
        return []

## 2c — DPLA API

In [4]:
def search_dpla(name, api_key, max_results=20):
    """
    Search DPLA for items mentioning a person's name.
    DPLA aggregates items from state libraries, historical societies, and universities.
    Free API key at dp.la/info/developers/codex/
    """
    if not api_key:
        return []
    
    try:
        resp = requests.get(
            "https://api.dp.la/v2/items",
            params={
                "q": f'"{name}"',
                "api_key": api_key,
                "page_size": max_results,
            },
            timeout=20
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"    DPLA error: {e}")
        return []
    
    results = []
    for doc in data.get("docs", []):
        sr = doc.get("sourceResource", {})
        titles = sr.get("title", [])
        title = titles[0] if isinstance(titles, list) and titles else str(titles)
        desc = sr.get("description", "")
        if isinstance(desc, list):
            desc = " ".join(desc)
        date_info = sr.get("date", {})
        if isinstance(date_info, list):
            date_info = date_info[0] if date_info else {}
        
        results.append({
            "item_id": doc.get("id", ""),
            "title": title,
            "description": desc[:300] if desc else "",
            "url": doc.get("isShownAt", ""),
            "provider": doc.get("dataProvider", ""),
            "date": date_info.get("displayDate", "") if isinstance(date_info, dict) else "",
            "type": sr.get("type", ""),
        })
    
    return results

## Run discovery loop

In [6]:
# Configuration
RUN_GOOGLE_BOOKS = bool(GOOGLE_BOOKS_KEY)
RUN_HATHITRUST   = False   # no key needed
RUN_DPLA         = bool(DPLA_KEY)
DRY_RUN          = False  # set True to preview without writing

DELAY_GOOGLE  = 0.5   # seconds between Google Books calls
DELAY_HATHI   = 1.0   # seconds between HathiTrust calls
DELAY_DPLA    = 0.3   # seconds between DPLA calls

print("Discovery configuration:")
print(f"  Google Books: {'ENABLED' if RUN_GOOGLE_BOOKS else 'DISABLED (no API key)'}")
print(f"  HathiTrust:   {'ENABLED' if RUN_HATHITRUST else 'DISABLED'}")
print(f"  DPLA:         {'ENABLED' if RUN_DPLA else 'DISABLED (no API key)'}")
print()

# Determine pending persons (those missing ALL three sub-steps)
steps_2 = ['2a_google_books', '2b_hathitrust', '2c_dpla']
todo_2a = set(pending_persons(conn, '2a_google_books')) if RUN_GOOGLE_BOOKS else set()
todo_2b = set(pending_persons(conn, '2b_hathitrust'))   if RUN_HATHITRUST   else set()
todo_2c = set(pending_persons(conn, '2c_dpla'))         if RUN_DPLA         else set()
todo_all = todo_2a | todo_2b | todo_2c
print(f"Pending — 2a: {len(todo_2a)}, 2b: {len(todo_2b)}, 2c: {len(todo_2c)}")
print(f"Total unique persons to process: {len(todo_all)}\n")

Discovery configuration:
  Google Books: ENABLED
  HathiTrust:   DISABLED
  DPLA:         ENABLED

Pending — 2a: 563, 2b: 0, 2c: 563
Total unique persons to process: 563



In [7]:
for research_id in tqdm(sorted(todo_all)):
    row = conn.execute(
        "SELECT canonical_name FROM persons WHERE research_id=?", (research_id,)
    ).fetchone()
    if not row:
        continue
    name = row["canonical_name"]
    variants = name_variants_for(conn, research_id)
    
    # 2a — Google Books
    if research_id in todo_2a:
        try:
            gb_results = []
            for search_name in ([name] + variants)[:2]:
                results = search_google_books(search_name, GOOGLE_BOOKS_KEY)
                for r in results:
                    if not DRY_RUN:
                        log_source(conn, research_id, 'google_books',
                                   title=r['title'], url=r['info_link'],
                                   item_id=r['item_id'],
                                   snippet=r['snippet'][:500] if r['snippet'] else None,
                                   relevance_note=f"{r['authors']} ({r['published_date']})",
                                   discovery_step='2a_google_books')
                    gb_results.append(r)
                time.sleep(DELAY_GOOGLE)
            if not DRY_RUN:
                set_progress(conn, research_id, '2a_google_books', 'done', result_count=len(gb_results))
        except Exception as e:
            if not DRY_RUN:
                set_progress(conn, research_id, '2a_google_books', 'failed', error_msg=str(e))
    
    # 2b — HathiTrust
    if research_id in todo_2b:
        try:
            ht_results = search_hathitrust_catalog(name)
            for r in ht_results:
                if not DRY_RUN:
                    log_source(conn, research_id, 'hathitrust',
                               title=r['title'], url=r['url'],
                               item_id=r['item_id'],
                               discovery_step='2b_hathitrust')
            if not DRY_RUN:
                set_progress(conn, research_id, '2b_hathitrust', 'done', result_count=len(ht_results))
            time.sleep(DELAY_HATHI)
        except Exception as e:
            if not DRY_RUN:
                set_progress(conn, research_id, '2b_hathitrust', 'failed', error_msg=str(e))
    
    # 2c — DPLA
    if research_id in todo_2c:
        try:
            dpla_results = search_dpla(name, DPLA_KEY)
            for r in dpla_results:
                if not DRY_RUN:
                    log_source(conn, research_id, 'dpla',
                               title=r['title'], url=r['url'],
                               item_id=r['item_id'],
                               snippet=r['description'],
                               relevance_note=f"{r['provider']} ({r['date']})",
                               discovery_step='2c_dpla')
            if not DRY_RUN:
                set_progress(conn, research_id, '2c_dpla', 'done', result_count=len(dpla_results))
            time.sleep(DELAY_DPLA)
        except Exception as e:
            if not DRY_RUN:
                set_progress(conn, research_id, '2c_dpla', 'failed', error_msg=str(e))

  0%|          | 0/563 [00:00<?, ?it/s]

 12%|█▏        | 66/563 [05:03<1:27:00, 10.50s/it]

    DPLA error: HTTPSConnectionPool(host='api.dp.la', port=443): Read timed out. (read timeout=20)


100%|██████████| 563/563 [42:12<00:00,  4.50s/it] 


## Post-discovery summary

In [8]:
print("=== Discovery Summary ===\n")

# Top books / volumes that mention multiple persons (highest-value targets for Step 3)
top_sources = conn.execute("""
    SELECT title, source_type, COUNT(DISTINCT research_id) as persons_mentioned,
           GROUP_CONCAT(DISTINCT item_id) as item_ids
    FROM sources_discovered
    WHERE title != '' AND title IS NOT NULL
    GROUP BY title, source_type
    HAVING persons_mentioned >= 2
    ORDER BY persons_mentioned DESC
    LIMIT 30
""").fetchall()

print("Top sources mentioning multiple persons in our dataset:")
print(f"{'Persons':>7}  {'Type':<15} {'Title'}")
print("-" * 70)
for r in top_sources:
    print(f"  {r['persons_mentioned']:5d}  {r['source_type']:<15} {(r['title'] or '')[:60]}")

print()
per_step = conn.execute("""
    SELECT discovery_step, COUNT(*) as n, COUNT(DISTINCT research_id) as persons
    FROM sources_discovered GROUP BY discovery_step
""").fetchall()
print("Sources per discovery step:")
for r in per_step:
    print(f"  {r['discovery_step']:<25} {r['n']:>6} sources  ({r['persons']} persons covered)")

=== Discovery Summary ===

Top sources mentioning multiple persons in our dataset:
Persons  Type            Title
----------------------------------------------------------------------
     85  google_books    Official Register of the United States
     65  google_books    House documents
     65  google_books    Report
     57  google_books    Geo. P. Rowell and Co.'s American Newspaper Directory
     56  google_books    Annual Report
     44  google_books    Register of Officers and Agents, Civil, Military and Naval [
     34  google_books    American Newspaper Directory
     34  google_books    Official Register
     25  google_books    The Trow City Directory Co.'s, Formerly Wilson's, Copartners
     22  google_books    Congressional Record
     22  google_books    Rowell's American Newspaper Directory
     21  google_books    Journal
     20  google_books    Cumulated Index Medicus
     19  google_books    House Documents
     19  google_books    Transactions
     17  google_books